<a href="https://colab.research.google.com/github/arlinrus/vkr/blob/main/diplom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data load

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
# %%capture
# !pip install --upgrade matplotlib numpy==1.26.0 # need refresh enviroment after installation

In [5]:
# %%capture
# !pip install basemap phik sweetviz

In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import seaborn as sns


In [7]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
data_purchases = pd.read_csv('/content/drive/MyDrive/Diplom/amazon-purchases.csv', na_values=['NaN'])

In [ ]:
data_purchases.info()

In [ ]:
data_purchases.dtypes

In [ ]:
data_purchases.isnull().sum()


# DATA PREPROCESSING

Преобразуем тип данных, название стообцов, и пропуски в датасете.

In [ ]:
df = data_purchases.copy()

df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)

df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['total_price'] = (df['purchase_price_per_unit'] * df['quantity'])

df = df.rename(columns={
    'survey_responseid':'user_id',
    'asin_isbn_product_code': 'asin_isbn',
})

df.columns

# new_col_names = [
#     re.sub(r"\s+", "_", s.strip().lower())
#     for s in data_purchases.columns
# ]

In [ ]:
cols = list(df.columns)
cols[0], cols[7] = cols[7], cols[0]
df = df[cols]

In [ ]:
df.head()

In [ ]:
missing_value = data_purchases.isna().sum()
ax = missing_value.plot(kind='bar', legend=False, color='skyblue', edgecolor='black', linestyle='-', rot=45, figsize=(12,6))
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', label_type='edge', padding=3, fontweight='bold')
plt.ylabel('Количество')
plt.tight_layout()
plt.show()

Обработаем пропуски в категориальных данных

In [ ]:
df['shipping_address_state'] = df['shipping_address_state'].fillna('uknown')
df['category'] = df['category'].fillna('uknown')
df['title'] = df['title'].fillna('uknown')
df['asin_isbn'] = df['asin_isbn'].fillna('uknown')


In [ ]:
df.isnull().sum()

In [ ]:
df.shape, df['user_id'].nunique(), df['category'].nunique(), df['order_date'].min(), df['order_date'].max()

## 3. Первичный EDA транзакционного уровня

На этом этапе анализируются сами транзакции: распределение цен, количества товаров, общей стоимости заказа, динамика покупок во времени и пропуски.

Определим правильно квартили в датасете

In [ ]:
num_cols_transaction = ['purchase_price_per_unit', 'quantity', 'total_price']
df[num_cols_transaction].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

In [ ]:
for col in num_cols_transaction:
    plt.figure(figsize=(9, 4))
    sns.histplot(np.log1p(df[col]), bins=60)
    plt.xlabel(f'log1p({col})')
    plt.show()


In [ ]:
month_orders = df.set_index('order_date').resample('M')['user_id'].count()
month_revenue = df.set_index('order_date').resample('M')['total_price'].sum()

Построим график количества транзакций и выручку за месяц

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

month_orders.plot(
    kind='line',
    color='red',
    grid=True,
    ax=ax[0]
)
ax[0].set_title('Orders by Month')
ax[0].set_ylabel('Количество транзакций')

month_revenue.plot(
    kind='line',
    color='red',
    grid=True,
    ax=ax[1]
)
ax[1].set_title('Revenue by Month')
ax[1].set_ylabel('Выручка')

plt.tight_layout()
plt.show()

In [ ]:
# сколько раз каждый пользователь покупал товар
count_user = df.groupby(['user_id', 'asin_isbn']).size().reset_index(name='product_purch')
repeat = count_user[(count_user['product_purch'] > 1) & (count_user['product_purch'] < 7)]
item = repeat['product_purch'].value_counts().sort_index()
pie_data = item[item.index > 5].copy()
pie_data['>5'] = item[item.index > 5].sum()

labels = [str(i) for i in pie_data.index]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

item.plot(
    kind='pie',
    ax=axes[0],
    autopct='%1.1f%%',
    startangle=90,
    title='Повторные покупки',
    ylabel='',
    colors=['olivedrab', 'rosybrown', 'gray', 'saddlebrown', 'skyblue']
)

item.plot(
    kind='pie',
    ax=axes[1],
    startangle=90,
    title='Повторные покупки',
    ylabel='',
    hatch=['**O', 'oO', 'O.O', '.||.']
)

plt.tight_layout()
plt.show()

In [ ]:
cutoff_date = pd.Timestamp('2021-01-01')
end_date = pd.Timestamp('2023-03-31')

df = df[df['order_date'] <= end_date]

past = df[df['order_date'] < cutoff_date].copy()
future = df[df['order_date'] >= cutoff_date].copy()
past.shape, future.shape, past['user_id'].nunique(), future['user_id'].nunique()

In [ ]:
ltv = (
    future.groupby('user_id')['total_price']
    .sum()
    .rename('ltv')
    .reset_index()
)

# Feature engineering

Сейчас произведем рассчеты и защитим некоторые столбцы от деления на 0

In [ ]:
def category_entropy(x):
  counts = x.value_counts(normalize=True)
  return - (counts * np.log1p(counts)).sum()

user = past.groupby('user_id').agg(
    total_revenue=('total_price', 'sum'),
    num_orders=('total_price', 'count'),
    total_quantity=('quantity', 'sum'),
    avg_order_value=('total_price', 'mean'),
    median_order_value=('total_price', 'median'),
    std_order_value=('total_price', 'std'),
    min_order_value=('total_price', 'min'),
    max_order_value=('total_price', 'max'),
    avg_unit_price=('purchase_price_per_unit', 'mean'),
    first_order_date=('order_date', 'min'),
    last_order_date=('order_date', 'max'),
    active_days=('order_date', lambda x: x.dt.date.nunique()),
    category_count=('category', 'nunique'),
    product_count=('asin_isbn', 'nunique'),
    state_count=('shipping_address_state', 'nunique'),
    category_entropy=('category', category_entropy)
).reset_index()

Рассчитаем временные признаки

In [ ]:
user['recency'] = (cutoff_date - user['last_order_date']).dt.days
user['customer_age'] = (cutoff_date - user['first_order_date']).dt.days #время с момента первого взаимодействия
user['tenure'] = (user['last_order_date'] -user['first_order_date']).dt.days #длительность активного периода

In [ ]:
#если покупатель ничего не покупает - это не значит, что у него ничего нету, поэтому вместо 0 у нас засчитывается как один день
user['zero_division_customer'] = user['customer_age'].replace(0, 1)
user['zero_division_tenure'] = user['tenure'].replace(0, 1)

In [ ]:
# Интенсивность поведения
user['purchase_frequency_rate'] = user['num_orders'] / user['zero_division_customer']
user['revenue_per_day'] = user['total_revenue'] / user['zero_division_customer']
user['orders_per_active_day'] = user['num_orders'] / user['active_days'].replace(0, 1)
user['active_days_ratio'] = user['active_days'] / user['zero_division_customer']
user['average_order_size'] = user['total_quantity'] / user['num_orders'].replace(0, 1)
user['recency_normalized'] = user['recency'] / user['zero_division_customer']


In [ ]:
user.head()

In [ ]:
user.columns

Рассчитаем интервалы между покупками

In [ ]:
past_n = past.sort_values(['user_id', 'order_date']).copy()
past_n['days_between_orders_'] = past_n.groupby('user_id')['order_date'].diff().dt.days

In [ ]:
past_n

In [ ]:
interval_features = past_n.groupby('user_id')['days_between_orders_'].agg(
    avg_days_between_orders='mean',
    median_days_between_orders='median',
    std_days_between_orders='std',
    min_days_between_orders='min',
    max_days_between_orders='max'
).reset_index()

user = user.merge(interval_features, on='user_id', how='left')
user.head()

In [ ]:
cols = [
    'avg_days_between_orders',
    'median_days_between_orders',
    'std_days_between_orders',
    'min_days_between_orders',
    'max_days_between_orders'
]

user[cols].hist(figsize=(12, 10), bins=30)
plt.tight_layout()
plt.show()

Признаки первых 30 дней после первой покупки помогают понять, насколько быстро пользователь начинает приносить ценность.

In [ ]:
first_day = past.groupby('user_id')['order_date'].min().rename('first_date')
past_early = past.merge(first_day, on='user_id', how='left')
past_early['days_from_start'] = (past_early['order_date'] - past_early['first_date']).dt.days

In [ ]:
early_30 = past_early[past_early['days_from_start'] <= 30]
early_features = early_30.groupby('user_id').agg(
    revenue_first_30d=('total_price', 'sum'),
    orders_first_30d=('total_price', 'count'),
    quantity_first_30d=('quantity', 'sum'),
    categories_first_30d=('category', 'nunique')
).reset_index()

early_features['avg_order_value_first_30d'] = (
    early_features['revenue_first_30d'] / early_features['orders_first_30d'].replace(0, np.nan)
)

user = user.merge(early_features, on='user_id', how='left')
user.head()

### 5.3 Доля топ-категории

Этот признак отражает специализацию покупок. Если большая часть заказов приходится на одну категорию, пользователь имеет более устойчивый категорийный паттерн.

In [ ]:
cat_counts = past.groupby(['user_id', 'category']).size().rename('cat_orders').reset_index()
top_cat = cat_counts.loc[cat_counts.groupby('user_id')['cat_orders'].idxmax()].copy()
total_orders_by_user = past.groupby('user_id').size().rename('num_orders_check').reset_index()

top_cat = top_cat.merge(total_orders_by_user, on='user_id', how='left')
top_cat['top_category_share'] = top_cat['cat_orders'] / top_cat['num_orders_check']
top_cat = top_cat[['user_id', 'category', 'top_category_share']].rename(columns={'category': 'top_category'})

user = user.merge(top_cat, on='user_id', how='left')
user.head()

### 5.4 Тренды активности

Сравниваем последние 90 дней исторического периода с предыдущими 90 днями. Это позволяет увидеть рост или падение активности до даты отсечения.

In [ ]:
last_90_start = cutoff_date - pd.Timedelta(days=90)
prev_90_start = cutoff_date - pd.Timedelta(days=180)

last_90 = past[(past['order_date'] >= last_90_start) & (past['order_date'] < cutoff_date)]
prev_90 = past[(past['order_date'] >= prev_90_start) & (past['order_date'] < last_90_start)]

last_90_features = last_90.groupby('user_id').agg(
    revenue_last_90d=('total_price', 'sum'),
    orders_last_90d=('total_price', 'count')
).reset_index()

prev_90_features = prev_90.groupby('user_id').agg(
    revenue_prev_90d=('total_price', 'sum'),
    orders_prev_90d=('total_price', 'count')
).reset_index()

user = user.merge(last_90_features, on='user_id', how='left')
user = user.merge(prev_90_features, on='user_id', how='left')

for col in ['revenue_last_90d', 'orders_last_90d', 'revenue_prev_90d', 'orders_prev_90d']:
    user[col] = user[col].fillna(0)

user['revenue_trend_90d'] = (user['revenue_last_90d'] + 1) / (user['revenue_prev_90d'] + 1)
user['orders_trend_90d'] = (user['orders_last_90d'] + 1) / (user['orders_prev_90d'] + 1)

user.head()

In [ ]:
from scipy.stats import gaussian_kde

cols = [
    'revenue_last_90d',
    'orders_last_90d',
    'revenue_prev_90d',
    'orders_prev_90d'
]

data = user[cols].copy()

fig, ax = plt.subplots(figsize=(10, 5))

offset = 1.2
x_min = data.min().min()
x_max = data.max().max()
x = np.linspace(x_min, x_max, 500)

for i, col in enumerate(data.columns):
    values = data[col].dropna()
    kde = gaussian_kde(values)
    y = kde(x)
    y = y / y.max()

    baseline = i * offset

    ax.fill_between(x, baseline, baseline + y, alpha=0.65)
    ax.plot(x, baseline + y, linewidth=1)
    ax.hlines(baseline, x_min, x_max, linewidth=0.5)

ax.set_yticks(np.arange(len(data.columns)) * offset)
ax.set_yticklabels(data.columns)

ax.set_xlabel("value")
ax.set_ylabel("feature")
ax.set_title("Ridgeline plot")

plt.tight_layout()
plt.show()

In [ ]:
user["revenue_last_90d"].quantile([0.5, 0.9, 0.95, 0.99])

In [ ]:
user["revenue_delta"] = user["revenue_last_90d"] - user["revenue_prev_90d"]
user["orders_delta"] = user["orders_last_90d"] - user["orders_prev_90d"]

### 5.5 Бинарные поведенческие флаги

In [ ]:
user['is_repeat_buyer'] = (user['num_orders'] > 1).astype(int)
user['is_recent_buyer_90d'] = (user['recency'] <= 90).astype(int)
user['has_many_categories'] = (user['category_count'] > user['category_count'].median()).astype(int)

user.head()

## 6. Итоговый датасет для моделирования

In [ ]:
data = user.merge(ltv, on='user_id', how='left')
data['ltv'] = data['ltv'].fillna(0)
data['ltv_log'] = np.log1p(data['ltv'])

date_cols = ['first_order_date', 'last_order_date']

num_cols = data.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    if col not in ['ltv', 'ltv_log']:
        data[col] = data[col].replace([np.inf, -np.inf], np.nan)
        data[col] = data[col].fillna(0)

for col in ['top_category']:
    data[col] = data[col].fillna('unknown')

data.shape

In [ ]:
data[['ltv', 'ltv_log']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(data['ltv'], bins=60, kde=True)
# plt.title('Распределение LTV')
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(data['ltv_log'], bins=60, kde=True)
# plt.title('Распределение log1p(LTV)')
plt.show()

In [ ]:
important_features = [
    'total_revenue', 'num_orders', 'avg_order_value', 'median_order_value', 'std_order_value',
    'recency', 'customer_age', 'tenure', 'purchase_frequency_rate', 'revenue_per_day',
    'avg_days_between_orders', 'std_days_between_orders', 'category_count', 'product_count',
    'top_category_share', 'revenue_first_30d', 'orders_first_30d',
    'revenue_last_90d', 'orders_last_90d', 'revenue_trend_90d', 'orders_trend_90d'
]

available_important_features = [c for c in important_features if c in data.columns]

data[available_important_features + ['ltv', 'ltv_log']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

## Посчитаем коррелицию

In [ ]:
numeric_features = [c for c in data.select_dtypes(include=[np.number]).columns if c not in ['ltv', 'ltv_log']]
pearson_corr = data[numeric_features + ['ltv_log']].corr(method='pearson')['ltv_log'].sort_values(ascending=False)
spearman_corr = data[numeric_features + ['ltv_log']].corr(method='spearman')['ltv_log'].sort_values(ascending=False)

cor_table = pd.DataFrame({
    'pearson_corr_ltv_log': pearson_corr,
    'spearman_corr_ltv_log': spearman_corr
}).drop(index='ltv_log')

cor_table.sort_values('spearman_corr_ltv_log', ascending=False).head(30)

In [ ]:
corr_ = cor_table['spearman_corr_ltv_log'].abs().sort_values(ascending=False).head(20).index.tolist()

plt.figure(figsize=(12,6))
sns.heatmap(data[corr_ + ['ltv_log']].corr(method='spearman'), cmap='coolwarm', center=0, annot=True)
plt.show()


In [ ]:
corr_

# 7.2 Проверка мультиколлинеарности
Если признаки сильно коррелируют между собой, линейные модели могут быть нестабильны. Для деревьев это менее критично, но всё равно полезно понимать дублирование признаков.

In [ ]:
corr_matrix = data[numeric_features].corr(method='spearman').abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2', 0: 'abs_spearman_corr'})
    .sort_values('abs_spearman_corr', ascending=False)
)
high_corr_pairs.head(25)

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 20))

plot_features = corr_[:8]

for idx, feature in enumerate(plot_features):
    ax = axes[idx // 4, idx % 4]

    sns.scatterplot(
        x=np.log1p(data[feature]),
        y=data['ltv_log'],
        alpha=0.5,
        ax=ax
    )

    sns.regplot(
        x=np.log1p(data[feature]),
        y=data['ltv_log'],
        scatter=False,
        lowess=True,
        ax=ax
    )

    ax.set_xlabel(f'log1p({feature})')
    ax.set_ylabel('log1p(LTV)')
    ax.set_title(f'Зависимость LTV от {feature}')

plt.tight_layout()
plt.show()

In [ ]:
top_categories = data['top_category'].value_counts().head(20)
top_categories

In [ ]:
category_ltv = (
    data.groupby('top_category')
    .agg(users=('user_id', 'count'), median_ltv=('ltv', 'median'), mean_ltv=('ltv', 'mean'), median_ltv_log=('ltv_log', 'median'))
    .query('users >= 20')
    .sort_values('median_ltv_log', ascending=False)
)
category_ltv.head(20)

# Machine learning

### предобработка

In [ ]:
%%capture

!pip install category_encoders
!pip install catboost
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from category_encoders.cat_boost import CatBoostEncoder

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.cluster import KMeans

from sklearn.linear_model import ElasticNet, Lasso, Ridge

from catboost import CatBoostRegressor, Pool

from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score, silhouette_score

In [ ]:
exclude = ['user_id', 'ltv', 'ltv_log', 'first_order_date', 'last_order_date']
features = [c for c in data.columns if c not in exclude]

X = data[features].copy()
y = data['ltv_log'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state= 42)

numeric_data = X.select_dtypes(include=[np.number]).columns.tolist()
text_data = X.select_dtypes(exclude=[np.number]).columns.tolist()


In [ ]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

Рассмотрим распределение целевой переменной

In [ ]:
y.plot(kind='kde', title='Target density');

Dummy Regressor — базовая модель, которая поможет сравнивать модели для проверки качества.




Обработаем числовые признаки с помощью StandartScaler и SimpleImputer

In [ ]:
numeric_pipline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

X_train_num = pd.DataFrame(
    numeric_pipline.fit_transform(X_train[numeric_data]),
    columns = numeric_data,
    index=X_train.index
)

X_test_num = pd.DataFrame(
    numeric_pipline.transform(X_test[numeric_data]),
    columns=numeric_data,
    index=X_test.index
)

In [ ]:
clusters = [
    'recency',
    'num_orders',
    'avg_order_value',
    'purchase_frequency_rate',
    'avg_days_between_orders',
    'revenue_per_day',
    'category_count'
]

In [ ]:
clusters = [k for k in clusters if k in X_train_num.columns]
silhouette_scores = {}
for c in range(2, 9):
    kmeans = KMeans(n_clusters=c, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_train_num[clusters])
    silhouette_scores[c] = silhouette_score(X_train_num[clusters], labels)

silhouette_scores

In [ ]:
best_k = max(silhouette_scores, key=silhouette_scores.get)
best_k

Обучим финальный K-means

In [ ]:
kmeans = KMeans(
    n_clusters = best_k,
    random_state=42,
    n_init=10
)
X_train_cl = kmeans.fit_predict(X_train_num[clusters])
X_test_cl = kmeans.predict(X_test_num[clusters])

In [ ]:
X_train_num_copy = X_train_num.copy()
X_train_num_copy['cluster'] = X_train_cl

X_train_num_copy.groupby('cluster')[clusters].mean()

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2, random_state=42)

X_pca = pca.fit_transform(X_train_num[clusters])

plt.figure(figsize=(8,6))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=X_train_cl,
    alpha=0.6
)

plt.xlabel('PC1')
plt.ylabel('PC2')
# plt.title('K-Means clusters (PCA projection)')
plt.colorbar(scatter)
plt.show()

In [ ]:
cluster_profile = X_train_num_copy.groupby('cluster')[clusters].mean()
cluster_profile

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

scaler = MinMaxScaler()
scaled_profile = pd.DataFrame(
    scaler.fit_transform(cluster_profile),
    columns=cluster_profile.columns,
    index=cluster_profile.index
)

In [ ]:
categories = list(scaled_profile.columns)
N = len(categories)

angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig = plt.figure(figsize=(8,8))
ax = plt.subplot(111, polar=True)

for cluster in scaled_profile.index:
    values = scaled_profile.loc[cluster].tolist()
    values += values[:1]
    ax.plot(angles, values, label=f'Cluster {cluster}')
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
plt.legend()
# plt.title('Behavioral cluster profiles')
plt.show()

In [ ]:
train_cluster_df = pd.get_dummies(
    X_train_cl,
    prefix='cluster',
    dtype=int
)

test_cluster_df = pd.get_dummies(
    X_test_cl,
    prefix='cluster',
    dtype=int
)

In [ ]:
train_cluster_df.index = X_train.index
test_cluster_df.index = X_test.index

In [ ]:
train_cluster_df, test_cluster_df = train_cluster_df.align(
    test_cluster_df,
    join='left',
    axis=1,
    fill_value=0
)

In [ ]:
cat_encoder = CatBoostEncoder(cols=['top_category'])

X_train_cat = cat_encoder.fit_transform(
    X_train[['top_category']],
    y_train
)

X_test_cat = cat_encoder.transform(
    X_test[['top_category']]
)

In [ ]:
X_train_model = pd.concat(
    [X_train_num, X_train_cat, train_cluster_df],
    axis=1
)

X_test_model = pd.concat(
    [X_test_num, X_test_cat, test_cluster_df],
    axis=1
)

In [ ]:
dm = DummyRegressor(strategy='mean')

dm.fit(X_train_model, y_train)
y_pred_dummy = dm.predict(X_test_model)

mae = mean_absolute_error(y_test, y_pred_dummy)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_dummy))
r2 = r2_score(y_test, y_pred_dummy)

print('MAE:', mae)
print('RMSE:', rmse)
print('R2:', r2)

### модели

In [ ]:
lr = LinearRegression()

lr.fit(X_train_model, y_train)
y_pred_lr = lr.predict(X_test_model)

mae = mean_absolute_error(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2 = r2_score(y_test, y_pred_lr)

print('MAE:', mae)
print('RMSE:', rmse)
print('R2:', r2)

In [ ]:
ridge = Ridge(alpha=1.0)

ridge.fit(X_train_model, y_train)

y_pred_ridge = ridge.predict(X_test_model)

mae = mean_absolute_error(y_test, y_pred_ridge)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
r2 = r2_score(y_test, y_pred_ridge)

print('Ridge')
print('MAE:', mae)
print('RMSE:', rmse)
print('R2:', r2)

In [ ]:
lasso = Lasso(alpha=0.001, max_iter=10000)

lasso.fit(X_train_model, y_train)

y_pred_lasso = lasso.predict(X_test_model)

mae = mean_absolute_error(y_test, y_pred_lasso)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
r2 = r2_score(y_test, y_pred_lasso)

print('Lasso')
print('MAE:', mae)
print('RMSE:', rmse)
print('R2:', r2)

In [ ]:
import matplotlib.pyplot as plt
def make_graph(actual, prediction, name):
    plt.figure(figsize=(8, 6))

    results = pd.DataFrame({
        'True': actual,
        'Prediction': prediction
    })

    sns.scatterplot(
        x='True',
        y='Prediction',
        data=results,
        alpha=0.5
    )

    d_line = np.arange(
        results.min().min(),
        results.max().max()
    )

    plt.plot(
        d_line,
        d_line,
        color='red',
        linestyle='--'
    )

    plt.xlabel('True values')
    plt.ylabel('Predicted values')
    plt.grid(True)
    plt.show()

In [ ]:
make_graph(y_test, y_pred_dummy, name='dummy')

In [ ]:
make_graph(y_test, y_pred_lr, name='Linear Regression')

In [ ]:
make_graph(y_test, y_pred_lasso, name='Lasso')

In [ ]:
elastic = ElasticNet(random_state=42, max_iter=10000)

elastic_params = {
    'alpha': [0.0001, 0.001, 0.01, 0.1, 1],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

elastic_search = GridSearchCV(
    elastic,
    elastic_params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

elastic_search.fit(X_train_model, y_train)
elastic_best = elastic_search.best_estimator_
elastic_pred = elastic_best.predict(X_test_model)



In [ ]:
mae = mean_absolute_error(y_test, elastic_pred)
rmse = np.sqrt(mean_squared_error(y_test, elastic_pred))
r2 = r2_score(y_test, elastic_pred)

print('Best params:', elastic_search.best_params_)
print('MAE:', mae)
print('RMSE:', rmse)
print('R2:', r2)

In [ ]:
ridge = Ridge(random_state=42)

ridge_params = {
    'alpha': [0.01, 0.1, 1, 10, 50, 100]
}

ridge_search = GridSearchCV(
    ridge,
    ridge_params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

ridge_search.fit(X_train_model, y_train)

ridge_best = ridge_search.best_estimator_
ridge_pred = ridge_best.predict(X_test_model)

In [ ]:
lasso = Lasso(random_state=42, max_iter=10000)

lasso_params = {
    'alpha': [0.0001, 0.001, 0.01, 0.1, 1]
}

lasso_search = GridSearchCV(
    lasso,
    lasso_params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

lasso_search.fit(X_train_model, y_train)

lasso_best = lasso_search.best_estimator_
lasso_pred = lasso_best.predict(X_test_model)

In [ ]:
results = pd.DataFrame({
    'Model': ['LinearRegression', 'Ridge', 'Lasso', 'ElasticNet'],
    'R2': [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, ridge_pred),
        r2_score(y_test, lasso_pred),
        r2_score(y_test, elastic_pred)
    ],
    'RMSE': [
        root_mean_squared_error(y_test, y_pred_lr),
        root_mean_squared_error(y_test, ridge_pred),
        root_mean_squared_error(y_test, lasso_pred),
        root_mean_squared_error(y_test, elastic_pred)
    ],
    'MAE': [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, ridge_pred),
        mean_absolute_error(y_test, lasso_pred),
        mean_absolute_error(y_test, elastic_pred)
    ]
})

results.sort_values('R2', ascending=False)

### Теперь проверим **Random Forest**,**XGBoost** (Extreme Gradient Boosting) и CatBoost

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

CatBoost

In [ ]:
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

train_pool = Pool(
    X_train_model,
    y_train
)

test_pool = Pool(
    X_test_model,
    y_test
)

In [ ]:
from sklearn.model_selection import ParameterSampler

param_grid = {
    "depth": [4, 5, 6, 7, 8, 9, 10],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "l2_leaf_reg": [1, 3, 5, 7, 9, 12],
    "bagging_temperature": [0, 0.5, 1, 3, 5],
    "random_strength": [0.5, 1, 2, 5, 10],
    "subsample": [0.7, 0.8, 0.9, 1.0],
}

best_rmse = np.inf
best_model = None
best_params = None

for params in ParameterSampler(param_grid, n_iter=120, random_state=42):
    model = CatBoostRegressor(
        iterations=3000,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        early_stopping_rounds=150,
        verbose=0,
        **params
    )

    model.fit(
        train_pool,
        eval_set=test_pool,
        use_best_model=True
    )

    pred = model.predict(X_test_model)
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    if rmse < best_rmse:
        best_rmse = rmse
        best_model = model
        best_params = params

In [ ]:
print("Best params:", best_params)
print("Best RMSE:", best_rmse)
print("Best R2:", r2_score(y_test, best_model.predict(X_test_model)))

In [ ]:
cat_model = best_model
y_pred = cat_model.predict(X_test_model)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("CatBoost results on ltv_log:")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

feature_importance = pd.DataFrame({
    "feature": X_train_model.columns,
    "importance": cat_model.get_feature_importance()
}).sort_values(by="importance", ascending=False)

feature_importance.head(20)

RandomForest

In [ ]:

def evaluate_model(name, y_test, y_pred):
    return {
        'model': name,
        'MSE': mean_squared_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE': mean_absolute_error(y_test, y_pred),
        'R2': r2_score(y_test, y_pred)
    }

In [ ]:
rf = RandomForestRegressor(random_state=2023)
parameters = {
    'n_estimators': [300, 500, 800, 1200],
    'max_depth': [8, 12, 16, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 0.5, 0.7],
    'bootstrap': [True],
    'max_samples': [0.7, 0.8, 0.9, None]
}

rf_classifier = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=2023, n_jobs=-1),
    param_distributions=parameters,
    n_iter=60,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=2,
    random_state=2023
)


rf_classifier.fit(X_train_model, y_train)


In [ ]:
y_preds_param_rf = rf_classifier.best_estimator_.predict(X_test_model)

print('Best Random Forest parameters:', rf_classifier.best_params_)


In [ ]:
print('R2:', r2_score(y_test, y_preds_param_rf))
print('RMSE:', root_mean_squared_error(y_test, y_preds_param_rf))
print('MSE:', mean_squared_error(y_test, y_preds_param_rf))
print('MAE:', mean_absolute_error(y_test, y_preds_param_rf))

XGB Regressor

In [ ]:
xgb = XGBRegressor(objective='reg:squarederror')
xgb.fit(X_train_model, y_train)
y_preds_xgd = xgb.predict(X_test_model)

In [ ]:
print('RMSE:', root_mean_squared_error(y_test, y_preds_xgd))
print('MSE:', mean_squared_error(y_test, y_preds_xgd))
print('MAE:', mean_absolute_error(y_test, y_preds_xgd))
print('R2:', r2_score(y_test, y_preds_xgd))

In [ ]:
model = XGBRegressor()
parameters = {
    'n_estimators': [300, 500, 800, 1200, 1500],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6, 8, 10],
    'min_child_weight': [1, 3, 5, 7, 10],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.01, 0.05, 0.1, 0.3],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0.5, 1, 3, 5, 10]
}

xgb_random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=parameters,
    n_iter=60,
    cv=2,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

xgb_random_search.fit(X_train_model, y_train)

y_preds_xgdr = xgb_random_search.best_estimator_.predict(X_test_model)

In [ ]:
y_preds_xgdr = xgb_random_search.best_estimator_.predict(X_test)

In [ ]:
print('RMSE:', root_mean_squared_error(y_test, y_preds_xgdr))
print('MSE:', mean_squared_error(y_test, y_preds_xgdr))
print('MAE:', mean_absolute_error(y_test, y_preds_xgdr))
print('R2:', r2_score(y_test, y_preds_xgdr))

In [ ]:
joblib.dump(xgb_random_search.best_estimator_, 'xgb_best_model.pkl')

In [ ]:
import joblib

joblib.dump(
    rf_classifier.best_estimator_,
    'rf_classifier.pkl'
)

In [ ]:
joblib.dump(rf_classifier.best_estimator, 'xgb_best_model.pkl')


In [ ]:
import joblib
joblib.dump(xgb_random_search.best_estimator_, 'xgb_best_model.pkl')

In [ ]:
joblib.dump(rf_classifier.best_estimator_, 'rf_classifie.pkl')

In [ ]:
import joblib

joblib.dump(
    xgb_random_search.best_estimator_,
    '/content/drive/MyDrive/xgb_best_model.pkl'
)

In [ ]:
import joblib

joblib.dump(
   rf_classifier.best_estimator_, '/content/drive/MyDrive/rf_classifie.pkl')

# Assesment of features